# HW1: Solving the N-Puzzle Using A*

In [1]:
import random
import importlib
import pandas as pd
from tqdm.auto import tqdm

import state, frontier, search
importlib.reload(state); importlib.reload(frontier); importlib.reload(search)

CAP = 50000
RUNS = 100


## Harness

In [2]:
# hdistance1 = tiles out of place
# hdistance2 = manhattan distance

ALGOS = [
    ("Uniform",    lambda x: 0 + state.path_len(x)),
    ("Greedy",     lambda x: state.hdistance2(x)),                        # h only, no g
    ("Tiles",      lambda x: state.hdistance1(x) + state.path_len(x)),
    ("Manhattan",  lambda x: state.hdistance2(x) + state.path_len(x)),
    ("Weighted",   lambda x: 4*state.hdistance2(x) + state.hdistance1(x) + state.path_len(x)),
    ("Linear_Con", lambda x: state.hdistance2(x) + 2*state.linear_conflicts(x) + state.path_len(x)),
]
ALGOS_DICT = dict(ALGOS)

def run_benchmark(n, algo_names, runs=RUNS, cap=CAP, seed=0):
    """
    Generates `runs` random nxn boards (seeded, so results reproduce) and runs every
    algorithm named in algo_names on each board. Returns {name: [(solved,pushed,popped,cost,path), ...]}.
    """
    random.seed(seed)
    results = {name: [] for name in algo_names}
    for _ in tqdm(range(runs), desc=f"{n}x{n} [{len(algo_names)} heur]", leave=False):
        board = state.create(n)[0]
        for name in algo_names:
            r = search.search(n, ALGOS_DICT[name], cap=cap, board=board[:])
            results[name].append(r)
    return results

def summarize(results):
    """Builds the Heuristic / Pushed / Popped / Path_Cost / Failure% dataframe."""
    rows = []
    for name, runs in results.items():
        pushed  = [r[1] for r in runs]
        popped  = [r[2] for r in runs]
        cost    = [r[3] for r in runs]      # literal: timed-out runs contribute 100
        failed  = [not r[0] for r in runs]
        rows.append({
            "Heuristic": name,
            "Pushed_(Ave)": sum(pushed)/len(pushed),
            "Popped_(Ave)": sum(popped)/len(popped),
            "Pushed_(Max)": max(pushed),
            "Popped_(Max)": max(popped),
            "Path_Cost_(Ave)": sum(cost)/len(cost),
            "Failure%": 100*sum(failed)/len(failed),
        })
    return pd.DataFrame(rows).round(2)


## Summary tables

All six heuristics, 3x3 and 4x4 (100 runs each, cap=150000); Manhattan and Linear_Con only for
the 5x5 (as the assignment allows skipping the rest there). Note that `Path_Cost_(Ave)` is
the **literal** average including timed-out runs at cost 100 (matches the assignment's step-1
instruction, not the sample table, which appears to average successes only).

In [3]:
ALL_NAMES = [name for name, _ in ALGOS]

results_3x3 = run_benchmark(3, ALL_NAMES, runs=RUNS, cap=CAP, seed=10)
table_3x3 = summarize(results_3x3)
table_3x3


3x3 [6 heur]:   0%|          | 0/100 [00:00<?, ?it/s]

,Heuristic,Pushed_(Ave),Popped_(Ave),Pushed_(Max),Popped_(Max),Path_Cost_(Ave),Failure%
0,Uniform,749.34,444.48,38362,23018,5.63,0.0
1,Greedy,2515.53,1440.80,50002,29412,10.93,5.0
2,Tiles,33.19,18.93,771,455,5.63,0.0
3,Manhattan,16.98,9.87,203,126,5.63,0.0
4,Weighted,21.98,12.58,228,128,5.99,0.0
5,Linear_Con,15.26,8.79,102,63,5.63,0.0


In [4]:
results_4x4 = run_benchmark(4, ALL_NAMES, runs=RUNS, cap=CAP, seed=20)
table_4x4 = summarize(results_4x4)
table_4x4


4x4 [6 heur]:   0%|          | 0/100 [00:00<?, ?it/s]

,Heuristic,Pushed_(Ave),Popped_(Ave),Pushed_(Max),Popped_(Max),Path_Cost_(Ave),Failure%
0,Uniform,38980.70,18292.56,50003,23506,74.50,72.0
1,Greedy,18053.12,8241.68,50003,25715,47.14,36.0
2,Tiles,7386.63,3455.72,50003,23935,22.54,9.0
3,Manhattan,1250.18,599.87,46939,22658,16.03,0.0
4,Weighted,2462.22,1152.09,42974,19996,21.71,0.0
5,Linear_Con,383.22,183.04,9438,4511,16.03,0.0


In [5]:
results_5x5 = run_benchmark(5, ["Manhattan", "Linear_Con"], runs=RUNS, cap=CAP, seed=30)
table_5x5 = summarize(results_5x5)
table_5x5


5x5 [2 heur]:   0%|          | 0/100 [00:00<?, ?it/s]

,Heuristic,Pushed_(Ave),Popped_(Ave),Pushed_(Max),Popped_(Max),Path_Cost_(Ave),Failure%
0,Manhattan,22143.44,9524.27,50003,22757,52.54,33.0
1,Linear_Con,16742.76,7212.58,50003,22451,46.18,23.0


## **A. Is the weighted heuristic (`4*hdistance2 + hdistance1`) admissible?**

No, it has a higher average sokoution length than manhattan, meaning it is not alwssy finding the optimal solution

## Data for thinking question B

In [11]:
def run_mdlc(n, ks, runs=RUNS, cap=CAP, seed=0):
    random.seed(seed)
    names = [f"MD+{k}*LC" for k in ks]
    results = {name: [] for name in names}
    for _ in tqdm(range(runs), desc=f"{n}x{n} MD+k*LC", leave=False):
        board = state.create(n)[0]
        for k, name in zip(ks, names):
            val_fn = (lambda x, k=k: state.md_plus_k_lc(x, k) + state.path_len(x))
            r = search.search(n, val_fn, cap=cap, board=board[:])
            results[name].append(r)
    return results

ks = [1, 2, 4]
mdlc_4x4 = summarize(run_mdlc(4, ks, runs=RUNS, cap=CAP, seed=50))
mdlc_4x4
mdlc_5x5 = summarize(run_mdlc(5, ks, runs=RUNS, cap=CAP, seed=50))
mdlc_5x5

4x4 MD+k*LC:   0%|          | 0/100 [00:00<?, ?it/s]

5x5 MD+k*LC:   0%|          | 0/100 [00:00<?, ?it/s]

,Heuristic,Pushed_(Ave),Popped_(Ave),Pushed_(Max),Popped_(Max),Path_Cost_(Ave),Failure%
0,MD+1*LC,18492.93,7971.75,50003,22149,48.23,26.0
1,MD+2*LC,16935.26,7276.46,50003,21905,45.87,22.0
2,MD+4*LC,15049.87,6478.91,50003,22522,44.67,20.0


I can't use the 5x5 because it times out even on admissible outputs, however on 4x4 we can see that Manhattan distance + linear conflicts and Manhattan distance + 2*linear conflicts give the same average path length, while  Manhattan distance + 4*linear conflicts give a longer, so it is not admissible. We can also see based on averge pushed, that Manhattan distance + 4*linear conflicts is faster than Manhattan distance + 2*linear conflicts is faster than Manhattan distance + linear conflicts. It seems that Manhattan distance + 2*linear conflicts is the sweet spot of fastest while still being admissible.
